# 🗃️ f6_m00b_preparacion_app.ipynb
## Genera `meta_test_app.parquet` para la app Streamlit (Fase 7)

🎯 **Qué hace**: Fusiona `meta_test.parquet` + `X_test.parquet` + flags `_missing` de `X_test_prep.parquet`
en un único fichero enriquecido listo para la app, sin joins en tiempo real.

📋 **Requisitos**:
- `data/06_evaluacion/meta_test.parquet` (generado en f6_m00_preparacion)
- `data/05_modelado/X_test_prep.parquet` (generado en Fase 5)
- `data/05_modelado/X_test.parquet` (generado en Fase 5)

📤 **Genera**:
- `data/06_evaluacion/meta_test_app.parquet` — 6.725 × 34 cols

🔄 **Flujo**: f6_m00_preparacion → **f6_m00b_preparacion_app** → app/utils/loaders.py

➡️ **Siguiente**: `loaders.py` carga este fichero con `RUTAS["meta_test_app"]`

In [1]:
# ── Celda 1: setup ────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# ROOT: subir niveles hasta encontrar src/
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

print(f"ROOT: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

ROOT: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
Python: 3.11.14


In [2]:
# ── Celda 2: cargar ficheros fuente ───────────────────────────────────────────
RUTA_META   = ROOT / "data" / "06_evaluacion" / "meta_test.parquet"
RUTA_X_PREP = ROOT / "data" / "05_modelado"   / "X_test_prep.parquet"
RUTA_X_RAW  = ROOT / "data" / "05_modelado"   / "X_test.parquet"

for ruta in [RUTA_META, RUTA_X_PREP, RUTA_X_RAW]:
    assert ruta.exists(), f"❌ No encontrado: {ruta}"

meta   = pd.read_parquet(RUTA_META)
X_prep = pd.read_parquet(RUTA_X_PREP)
X_raw  = pd.read_parquet(RUTA_X_RAW)

# Verificar índices idénticos
assert list(meta.index) == list(X_prep.index) == list(X_raw.index), \
    "❌ Los índices no coinciden — re-ejecuta f6_m00_preparacion.ipynb"

print(f"meta_test:   {meta.shape}")
print(f"X_test_prep: {X_prep.shape}")
print(f"X_test:      {X_raw.shape}")
print("✅ Ficheros cargados e índices verificados")

meta_test:   (6725, 13)
X_test_prep: (6725, 27)
X_test:      (6725, 27)
✅ Ficheros cargados e índices verificados


In [3]:
# ── Celda 3: construir meta_test_app ──────────────────────────────────────────
# Columnas ya en meta (no duplicar)
cols_meta = set(meta.columns)

# Features de X_test originales — excluir las que ya están en meta
# y excluir _missing (se añaden desde X_prep para consistencia)
cols_nuevas  = [c for c in X_raw.columns
                if c not in cols_meta and not c.endswith("_missing")]

# Flags _missing desde X_prep
missing_cols = [c for c in X_prep.columns if c.endswith("_missing")]

# Fusión por índice (segura porque índices son idénticos)
df_app = pd.concat([meta, X_raw[cols_nuevas], X_prep[missing_cols]], axis=1)

# Verificar sin duplicados
assert df_app.columns.duplicated().sum() == 0, "❌ Columnas duplicadas"

print(f"Shape final: {df_app.shape}")
print(f"Columnas ({len(df_app.columns)}):")
print(df_app.columns.tolist())
print()
nulos = df_app.isnull().sum()
nulos_reales = nulos[nulos > 0]
if len(nulos_reales):
    print("Nulos por columna:")
    print(nulos_reales)
else:
    print("✅ Sin nulos inesperados")

Shape final: (6725, 34)
Columnas (34):
['titulacion', 'rama', 'sexo', 'pais_nombre', 'provincia', 'via_acceso', 'abandono', 'per_id_ficticio', 'curso_aca_ini', 'vive_fuera', 'cupo', 'n_titulaciones', 'flag_cautela', 'cred_superados_anio_1er', 'nota_1er_anio', 'nota_acceso', 'nota_selectividad', 'tasa_abandono_titulacion', 'universidad_origen', 'n_anios_beca', 'anios_sin_beca', 'situacion_laboral', 'n_anios_trabajando', 'max_pagos', 'indicador_interrupcion', 'orden_preferencia', 'cred_repetidos', 'tasa_repeticion', 'edad_entrada', 'anios_gap', 'n_anios_sin_notas', 'nota_1er_anio_missing', 'nota_acceso_missing', 'nota_selectividad_missing']

Nulos por columna:
cupo    709
dtype: int64


In [4]:
# ── Celda 4: verificación de casos canónicos ──────────────────────────────────
# 5 perfiles seleccionados para la sección de ejemplos reales de la app
CASOS_CANONICOS = {
    "C1 — Ing. Informática (FP, abandona)":          15872,
    "C2 — Medicina (nota 11.94, no abandona)":       11906,
    "C3 — Comunicación (nota alta, abandona)":        14957,
    "C4 — Ing. Informática (mujer, no abandona)":     32472,
    "C5 — Derecho (mayor 25, trabaja, abandona)":      7176,
}

cols_check = ["titulacion", "rama", "sexo", "via_acceso",
              "nota_acceso", "nota_1er_anio", "edad_entrada",
              "n_anios_trabajando", "abandono"]

for nombre, idx in CASOS_CANONICOS.items():
    row = df_app.loc[idx]
    icono = "🔴 ABANDONA" if row["abandono"] == 1 else "🟢 NO ABANDONA"
    print(f"\n{icono} | {nombre}")
    for c in cols_check:
        if c in row.index:
            val = round(row[c], 3) if isinstance(row[c], float) else row[c]
            print(f"  {c}: {val}")


🔴 ABANDONA | C1 — Ing. Informática (FP, abandona)
  titulacion: Grado en Ingeniería Informática
  rama: TE
  sexo: Hombre
  via_acceso: Ciclo Formativo de Grado sup. o equivalente
  nota_acceso: 7.4
  nota_1er_anio: 6.42
  edad_entrada: 3.135
  n_anios_trabajando: 2
  abandono: 1

🟢 NO ABANDONA | C2 — Medicina (nota 11.94, no abandona)
  titulacion: Grado en Medicina
  rama: SA
  sexo: Mujer
  via_acceso: Pruebas acceso Bachiller Logse
  nota_acceso: 11.94
  nota_1er_anio: 7.54
  edad_entrada: 2.944
  n_anios_trabajando: 3
  abandono: 0

🔴 ABANDONA | C3 — Comunicación (nota alta, abandona)
  titulacion: Grado en Comunicación Audiovisual
  rama: SO
  sexo: Hombre
  via_acceso: Pruebas acceso Bachiller Logse
  nota_acceso: 8.57
  nota_1er_anio: 6.45
  edad_entrada: 2.996
  n_anios_trabajando: 4
  abandono: 1

🟢 NO ABANDONA | C4 — Ing. Informática (mujer, no abandona)
  titulacion: Grado en Ingeniería Informática
  rama: TE
  sexo: Mujer
  via_acceso: Pruebas acceso Bachiller Logse
  not

In [5]:
# ── Celda 5: guardar en data/06_evaluacion/ ───────────────────────────────────
RUTA_SALIDA = ROOT / "data" / "06_evaluacion" / "meta_test_app.parquet"
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)

df_app.to_parquet(RUTA_SALIDA, index=True)

print(f"✅ Guardado: {RUTA_SALIDA}")
print(f"   Shape:   {df_app.shape}")
print(f"   Tamaño:  {RUTA_SALIDA.stat().st_size / 1024:.1f} KB")

✅ Guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\06_evaluacion\meta_test_app.parquet
   Shape:   (6725, 34)
   Tamaño:  212.1 KB
